In [76]:
import RNA
import matplotlib.pyplot as plt
from draw_rna.ipynb_draw import draw_struct
import numpy as np

# Define input sequence and complementary sequences
Prom_nt = "TTCTAATACGACTCACTATA"
Prom_t = "TATAGTGAGTCGTATTA GAA"
O1_nt = "CTACATCCACATACTAATTAAC"
O1_t = "GTTAATTAGTATGTGGATGTAG"
i_1 = "GGGATG"
i_1_comp = "CATCCC"

# Alternative outputs if O1 output is not compatible
O2_nt = "CTACTTTCACTTCACAACATCA"
O2_t = "TGATGTTGTGAAGTGAAAGTAG"
O3_nt = "TACCATCACATTCAATAATCCT"
O3_t = "AGGATTATTGAATGTGATGGTA"

# Choose output

Output_nt = O3_nt
Output_t = O3_t

# Default insulation
default_insulation = i_1
default_insulation_comp = i_1_comp

i_2 = "GGGAGT"
i_2_comp = "ACTCCC"
i_3 = "GGGAGA"
i_3_comp = "TCTCCC"
i_4 = "GGGAAA"
i_4_comp = "TTTCCC"

# Alternative insulations
alternative_insulations = [i_2, i_3, i_4]
alternative_insulation_comps = [i_2_comp, i_3_comp, i_4_comp]

# Define threshold for basepair probability
bp_threshold=0.9

# Output file path
output_file = "dARTdesign_RNAoutputdetails.txt"

# Function to create the dART template strand
def create_dART_template(aptamer, insulation, insulation_comp):
    encoded_domain = f"{Output_t} {insulation} {aptamer} {insulation_comp}" # Change O1_nt to other output if necessary
    dART_template = f"{encoded_domain} {Prom_t}"
    
    return dART_template.replace(" ", ""), encoded_domain.replace(" ", "")
    
# Function to create encoded RNA transcript
def get_rna_transcript(encoded_sequence): 
    complement = {'A': 'U', 'T': 'A', 'U': 'A','C': 'G', 'G': 'C'}
    return ''.join(complement[base] for base in reversed(encoded_sequence))

aptamer = "UACGAUCCAGUGGGUUGAAGGAAAGUAACAGAUCGUA" # Insert aptamer sequence here.
aptamer_length = len(aptamer) # Check length of aptamer to ensure position of insulation domains and very the insulation domains are bound


def test_insulations_with_default(aptamer, default_insulation, default_insulation_comp, alternative_insulations, alternative_insulation_comps):
    with open(output_file, 'w') as file: # Writing in the details into txt file
    # Test the default insulation domains first
        insulations_to_test = [(default_insulation, default_insulation_comp)]
        # Add alternative insulations to the list
        insulations_to_test.extend(zip(alternative_insulations, alternative_insulation_comps))

        # Produces RNA based on encoded domain in the template 
        for insulation, insulation_comp in insulations_to_test:
            dART_template = create_dART_template(aptamer, insulation, insulation_comp)[0]
            encoded_RNA = create_dART_template(aptamer, insulation, insulation_comp)[1]
            # Generate the complementary RNA sequence
            rna_transcript = get_rna_transcript(encoded_RNA)
            # Fold RNA transcript at 37°C
            RNA.cvar.temperature = 37
            (structure, mfe) = RNA.fold(rna_transcript)
            fc = RNA.fold_compound(rna_transcript)
            fc.pf()
            basepair_probs = fc.bpp()
            start_region = range(7)
            target_region = range(6 + aptamer_length, 6 + aptamer_length + 8)
            binding_count = 0
    
            # Check if the binding count condition is met
            for i in start_region:
                for j in target_region:
                    prob = basepair_probs[i][j]
                    if prob > bp_threshold:
                        binding_count += 1
    
            # If the criteria are met, output the results and stop further testing
            if binding_count == 6:
                print("Order from IDT:")
                print("Recommended promoter non-template strand: ", Prom_nt + insulation)
                print("Recommended Output non-template strand: ", insulation_comp + Output_nt) # Change O1_nt to other output if necessary
                print("Recommended dART template strand sequence: ", dART_template)
                file.write("Recommended promoter non-template strand: " + Prom_nt + insulation + "\n")
                file.write("Recommended Output non-template strand: "+ insulation_comp + Output_nt + "\n") # Change O1_nt to other output if necessary
                file.write("Recommended dART template strand sequence: "+ dART_template + "\n")
                file.write("RNA Transcript: " + rna_transcript + "\n")
                file.write("Dot-Bracket Notation: " + structure + "\n")
                file.write(f"Minimum Free Energy (MFE): {mfe} kcal/mol\n")
                # draw_struct(rna_transcript, structure)  # OPTIONAL FEATURE - Draws secondary structure of RNA
                for i in start_region:
                    for j in target_region:
                        prob = basepair_probs[i][j]
                        # if prob > bp_threshold:
                            # print(f"Basepair probability({i},{j}) = {prob:.4f}")
                return  # Stop testing as a valid configuration is found

# Test the insulations
test_insulations_with_default(aptamer, default_insulation, default_insulation_comp, alternative_insulations, alternative_insulation_comps)


Order from IDT:
Recommended promoter non-template strand:  TTCTAATACGACTCACTATAGGGATG
Recommended Output non-template strand:  CATCCCTACCATCACATTCAATAATCCT
Recommended dART template strand sequence:  AGGATTATTGAATGTGATGGTAGGGATGUACGAUCCAGUGGGUUGAAGGAAAGUAACAGAUCGUACATCCCTATAGTGAGTCGTATTAGAA
